In [1]:
import polars as pl
import pathlib
import dataclasses
import pandas as pd
import functools
import sys

In [2]:
cwd = pathlib.Path.cwd()

In [3]:
sys.path.append(str(cwd.parent))

In [4]:
import uro_cbb.bracket as bracket

In [5]:
submission_probs = pd.read_csv(cwd.parent / 'data/kaggle_2026/cleaned/submissions/submission_base_mens.csv')
submission_probs

,ID,Pred
0,2026_1101_1102,0.738412
1,2026_1101_1103,0.118042
2,2026_1101_1104,0.026366
3,2026_1101_1105,0.626434
4,2026_1101_1106,0.613240
...,...,...
132128,2026_3478_3480,0.159065
132129,2026_3478_3481,0.438408
132130,2026_3479_3480,0.229781
132131,2026_3479_3481,0.551819


In [6]:
mens_team = pd.read_csv(cwd.parent / 'data/kaggle_2026/raw/MTeams.csv')
mens_team_spellings = pd.read_csv(cwd.parent / 'data/kaggle_2026/raw/MTeamSpellings.csv')

In [7]:
matchups_raw = [
    # EAST Region
    [1181, 1373],    # Duke (1) vs. Siena (16)
    [1326, 1395],    # Ohio State (8) vs. TCU (9)
    [1385, 1320],    # St. John's (5) vs. Northern Iowa (12)
    [1242, 1465],    # Kansas (4) vs. California Baptist (13)
    [1257, 1378],    # Louisville (6) vs. South Florida (11)
    [1277, 1295],    # Michigan State (3) vs. North Dakota State (14)
    [1417, 1416],    # UCLA (7) vs. UCF (10)
    [1163, 1202],    # UConn (2) vs. Furman (15)

    # MIDWEST Region
    [1276, 1224],    # Michigan (1) vs. Howard (16)
    [1208, 1387],    # Georgia (8) vs. Saint Louis (9)
    [1403, 1103],    # Texas Tech (5) vs. Akron (12)
    [1104, 1220],    # Alabama (4) vs. Hofstra (13)
    [1397, 1275],    # Tennessee (6) vs. Miami OH (11)
    [1438, 1460],    # Virginia (3) vs. Wright State (14)
    [1246, 1365],    # Kentucky (7) vs. Santa Clara (10)
    [1235, 1398],    # Iowa State (2) vs. Tennessee State (15)

    # SOUTH Region
    [1196, 1341],    # Florida (1) vs. Prairie View (16)
    [1155, 1234],    # Clemson (8) vs. Iowa (9)
    [1435, 1270],    # Vanderbilt (5) vs. McNeese (12)
    [1304, 1407],    # Nebraska (4) vs. Troy (13)
    [1314, 1433],    # UNC (6) vs. VCU (11)
    [1228, 1335],    # Illinois (3) vs. Penn (14)
    [1388, 1401],    # Saint Mary's (7) vs. Texas A&M (10)
    [1222, 1225],    # Houston (2) vs. Idaho (15)

    # WEST Region
    [1112, 1254],    # Arizona (1) vs. LIU (16)
    [1437, 1429],    # Villanova (8) vs. Utah State (9)
    [1458, 1219],    # Wisconsin (5) vs. High Point (12)
    [1116, 1218],    # Arkansas (4) vs. Hawaii (13)
    [1140, 1400],    # BYU (6) vs. Texas (11)
    [1211, 1244],    # Gonzaga (3) vs. Kennesaw State (14)
    [1274, 1281],    # Miami FL (7) vs. Missouri (10)
    [1345, 1474],    # Purdue (2) vs. Queens NC (15)
]

In [8]:
matchups = [
    [bracket.Team.from_id(matchup[0], mens_team), bracket.Team.from_id(matchup[1], mens_team)] 
    if matchup[0] < matchup[1]
    else [bracket.Team.from_id(matchup[1], mens_team), bracket.Team.from_id(matchup[0], mens_team)] 
    for matchup in matchups_raw
]

In [9]:
first_round = [
    bracket.Game(matchup[0], matchup[1], win_prob_df=submission_probs, year=2026) for matchup in matchups
]
second_round = [
    bracket.HyperGame(first_round[i], first_round[i+1], win_prob_df=submission_probs, year=2026)
    for i in range(0, len(first_round), 2)
]
third_round = [
    bracket.HyperGame(second_round[i], second_round[i+1], win_prob_df=submission_probs, year=2026)
    for i in range(0, len(second_round), 2)
]
fourth_round = [
    bracket.HyperGame(third_round[i], third_round[i+1], win_prob_df=submission_probs, year=2026)
    for i in range(0, len(third_round), 2)
]
fifth_round = [
    bracket.HyperGame(fourth_round[0], fourth_round[2], win_prob_df=submission_probs, year=2026),
    bracket.HyperGame(fourth_round[1], fourth_round[3], win_prob_df=submission_probs, year=2026)
]
sixth_round = [
    bracket.HyperGame(fifth_round[0], fifth_round[1], win_prob_df=submission_probs, year=2026),
]


In [16]:
print("PROB OF WINNING THE WHOLE TOURNAMENT")
for team, prob in sorted(sixth_round[0].all_win_probs.items(), key=lambda team_prob_tuple: -team_prob_tuple[1]):
    print(team, prob)

PROB OF WINNING THE WHOLE TOURNAMENT
Team(_id=1276, name='Michigan') 0.17721725315357997
Team(_id=1181, name='Duke') 0.1638696029825682
Team(_id=1112, name='Arizona') 0.12552579374658537
Team(_id=1345, name='Purdue') 0.08555874748804698
Team(_id=1228, name='Illinois') 0.0834552463107048
Team(_id=1196, name='Florida') 0.0734244579667374
Team(_id=1222, name='Houston') 0.0627925008968004
Team(_id=1235, name='Iowa St') 0.04711064163199411
Team(_id=1163, name='Connecticut') 0.028972316430254268
Team(_id=1435, name='Vanderbilt') 0.019907826496682095
Team(_id=1211, name='Gonzaga') 0.016001982629407115
Team(_id=1277, name='Michigan St') 0.013810347595859562
Team(_id=1104, name='Alabama') 0.013797018712801479
Team(_id=1438, name='Virginia') 0.011316307304994491
Team(_id=1116, name='Arkansas') 0.010313238706793392
Team(_id=1403, name='Texas Tech') 0.010045908818077822
Team(_id=1257, name='Louisville') 0.009591063402986436
Team(_id=1397, name='Tennessee') 0.00868776370318452
Team(_id=1385, name="

In [11]:
print("PROB OF MAKING IT TO SECOND ROUND")
for game in first_round:
    for team, prob in sorted(game.all_win_probs.items(), key=lambda team_prob_tuple: -team_prob_tuple[1])[:1]:
        print(team, prob)
    print()

PROB OF MAKING IT TO SECOND ROUND
Team(_id=1181, name='Duke') 0.9804966518244296

Team(_id=1326, name='Ohio St') 0.6931591717698508

Team(_id=1385, name="St John's") 0.8229447377868184

Team(_id=1242, name='Kansas') 0.8623272817697036

Team(_id=1257, name='Louisville') 0.761974807850407

Team(_id=1277, name='Michigan St') 0.9034567210820292

Team(_id=1417, name='UCLA') 0.6861164237007483

Team(_id=1163, name='Connecticut') 0.9596683198899696

Team(_id=1276, name='Michigan') 0.9851793165686178

Team(_id=1208, name='Georgia') 0.5318924219924231

Team(_id=1403, name='Texas Tech') 0.8215881209467837

Team(_id=1104, name='Alabama') 0.8853543339777057

Team(_id=1397, name='Tennessee') 0.8376270520543095

Team(_id=1438, name='Virginia') 0.9110802509137488

Team(_id=1365, name='Santa Clara') 0.5407310432670376

Team(_id=1235, name='Iowa St') 0.9760873244986782

Team(_id=1196, name='Florida') 0.988302620132108

Team(_id=1234, name='Iowa') 0.5866002974354962

Team(_id=1435, name='Vanderbilt') 0.

In [12]:
print("PROB OF MAKING IT TO SWEET 16")
for game in second_round:
    for team, prob in sorted(game.all_win_probs.items(), key=lambda team_prob_tuple: -team_prob_tuple[1]):
        print(team, prob)
    print()

PROB OF MAKING IT TO SWEET 16
Team(_id=1181, name='Duke') 0.8142143525726638
Team(_id=1326, name='Ohio St') 0.14875498908123969
Team(_id=1395, name='TCU') 0.035127238772518894
Team(_id=1373, name='Siena') 0.0019034195735776443

Team(_id=1385, name="St John's") 0.4772357460605496
Team(_id=1242, name='Kansas') 0.4544103805716857
Team(_id=1320, name='Northern Iowa') 0.04480813518426911
Team(_id=1465, name='Cal Baptist') 0.023545738183495608

Team(_id=1277, name='Michigan St') 0.5183462593679569
Team(_id=1257, name='Louisville') 0.401946118141851
Team(_id=1378, name='South Florida') 0.06621310032210982
Team(_id=1295, name='N Dakota St') 0.013494522168082279

Team(_id=1163, name='Connecticut') 0.6798495270113986
Team(_id=1417, name='UCLA') 0.24829774017860565
Team(_id=1416, name='UCF') 0.06782196990041145
Team(_id=1202, name='Furman') 0.004030762909584398

Team(_id=1276, name='Michigan') 0.8466044663694159
Team(_id=1208, name='Georgia') 0.0847342056307316
Team(_id=1387, name='St Louis') 0.0

In [13]:
print("PROB OF MAKING IT TO ELITE 8")
for game in third_round:
    for team, prob in sorted(game.all_win_probs.items(), key=lambda team_prob_tuple: -team_prob_tuple[1]):
        print(team, prob)
    print()

PROB OF MAKING IT TO ELITE 8
Team(_id=1181, name='Duke') 0.6333818578886683
Team(_id=1385, name="St John's") 0.14884843327119848
Team(_id=1242, name='Kansas') 0.1305232758018949
Team(_id=1326, name='Ohio St') 0.07066009606723431
Team(_id=1395, name='TCU') 0.010280464994740945
Team(_id=1320, name='Northern Iowa') 0.004517172204026613
Team(_id=1465, name='Cal Baptist') 0.0016475854532996533
Team(_id=1373, name='Siena') 0.0001411143189368811

Team(_id=1163, name='Connecticut') 0.3929602382940104
Team(_id=1277, name='Michigan St') 0.26642272246864496
Team(_id=1257, name='Louisville') 0.20190451417284863
Team(_id=1417, name='UCLA') 0.10335439593224413
Team(_id=1416, name='UCF') 0.017042947813944227
Team(_id=1378, name='South Florida') 0.016566433676076076
Team(_id=1295, name='N Dakota St') 0.001505553658561802
Team(_id=1202, name='Furman') 0.00024319398366968946

Team(_id=1276, name='Michigan') 0.6288306561778969
Team(_id=1104, name='Alabama') 0.173154429477333
Team(_id=1403, name='Texas Te

In [18]:
print("PROB OF MAKING IT TO FINAL 4")
for game in fourth_round:
    for team, prob in sorted(game.all_win_probs.items(), key=lambda team_prob_tuple: -team_prob_tuple[1]):
        print(team, prob)
    print()

PROB OF MAKING IT TO FINAL 4
Team(_id=1181, name='Duke') 0.4596453739080473
Team(_id=1163, name='Connecticut') 0.1685409891114669
Team(_id=1277, name='Michigan St') 0.10224817121465736
Team(_id=1257, name='Louisville') 0.07538307392817298
Team(_id=1385, name="St John's") 0.06958849964866815
Team(_id=1242, name='Kansas') 0.05691115419388121
Team(_id=1417, name='UCLA') 0.029959618745613897
Team(_id=1326, name='Ohio St') 0.028611566566112002
Team(_id=1416, name='UCF') 0.0028238407353015802
Team(_id=1378, name='South Florida') 0.0028190547567808685
Team(_id=1395, name='TCU') 0.0024157429480833135
Team(_id=1320, name='Northern Iowa') 0.000739702456102778
Team(_id=1465, name='Cal Baptist') 0.0001878656579822763
Team(_id=1295, name='N Dakota St') 0.000108738377993123
Team(_id=1202, name='Furman') 9.017109244984299e-06
Team(_id=1373, name='Siena') 7.590641891142571e-06

Team(_id=1276, name='Michigan') 0.45108839898265846
Team(_id=1235, name='Iowa St') 0.2059055557491404
Team(_id=1104, name='Al

In [17]:
print("PROB OF MAKING IT TO NATIONAL CHAMPIONSHIP")
for game in fifth_round:
    for team, prob in sorted(game.all_win_probs.items(), key=lambda team_prob_tuple: -team_prob_tuple[1]):
        print(team, prob)
    print()

PROB OF MAKING IT TO NATIONAL CHAMPIONSHIP
Team(_id=1181, name='Duke') 0.28223207129087724
Team(_id=1228, name='Illinois') 0.16014479032331388
Team(_id=1196, name='Florida') 0.14627890842841298
Team(_id=1222, name='Houston') 0.12703949510153847
Team(_id=1163, name='Connecticut') 0.07250192908329198
Team(_id=1435, name='Vanderbilt') 0.049785834709696816
Team(_id=1277, name='Michigan St') 0.039067338228611206
Team(_id=1257, name='Louisville') 0.027967338214011407
Team(_id=1385, name="St John's") 0.024262009459419087
Team(_id=1242, name='Kansas') 0.018250001178699456
Team(_id=1304, name='Nebraska') 0.015554165031681423
Team(_id=1417, name='UCLA') 0.008477304359615226
Team(_id=1326, name='Ohio St') 0.008406355714880664
Team(_id=1234, name='Iowa') 0.006329315079569803
Team(_id=1314, name='North Carolina') 0.004732081783811385
Team(_id=1388, name="St Mary's CA") 0.003959532839467378
Team(_id=1155, name='Clemson') 0.0017352129122562682
Team(_id=1401, name='Texas A&M') 0.0013416359945156657
Te